# AIC 2026 — Standard Preprocess — Single Part + Common

Notebook chuẩn dùng chung cho cả 4 thành viên.

Mỗi người chỉ sửa cấu hình `SOURCE_PART`, `PART_ROOT`, `ASSIGNED_GROUPS` và `COMMON_ROOT`.
Phần output luôn có **cùng tên file, cùng schema và cùng ý nghĩa**, để có thể merge trực tiếp sau này.

## Bộ output chuẩn

```text
processed/
├── manifest_keyframes.parquet
├── manifest_videos.parquet
├── feature_catalog.parquet
├── errors.csv
├── checksums.csv
├── summary.json
└── schema.json
```

`manifest_keyframes.parquet` là output chính cho retrieval. Mỗi dòng tương ứng với
một keyframe và đúng một hàng vector trong file CLIP `.npy`.


In [1]:
# Cell 1 — Cài thư viện cần thiết
!pip install -q pyarrow natsort tqdm

In [2]:
# Cell 2 — Import và cấu hình
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from natsort import natsorted
from tqdm.auto import tqdm

# =========================
# CHỈ SỬA KHU VỰC NÀY
# =========================
SOURCE_PART = "part2"  # part1 / part2 / part3 / part4
PART_ROOT = Path("/kaggle/input/datasets/minhdat27/aic2026-batch1-part2")
COMMON_ROOT = Path("/kaggle/input/datasets/minhdat27/aic2026-batch1-common")
ASSIGNED_GROUPS = ["L23", "L24", "L26_d", "L29"]
# =========================

OUTPUT_DIR = Path("/kaggle/working/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".mpeg", ".mpg"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
CALCULATE_SHA256 = True
MAX_VIDEOS: Optional[int] = None
MAX_KEYFRAMES: Optional[int] = None

CLIP_DIR = COMMON_ROOT / "clip-features-32-aic25-b1" / "clip-features-32"
MAP_DIR = COMMON_ROOT / "map-keyframes-aic25-b1" / "map-keyframes"
MEDIA_DIR = COMMON_ROOT / "media-info-aic25-b1" / "media-info"
OBJECTS_DIR = COMMON_ROOT / "objects-aic25-b1" / "objects"

for name, path in {
    "PART_ROOT": PART_ROOT,
    "COMMON_ROOT": COMMON_ROOT,
    "CLIP_DIR": CLIP_DIR,
    "MAP_DIR": MAP_DIR,
    "MEDIA_DIR": MEDIA_DIR,
    "OBJECTS_DIR": OBJECTS_DIR,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy {name}: {path}")

print("SOURCE_PART:", SOURCE_PART)
print("PART_ROOT:", PART_ROOT)
print("COMMON_ROOT:", COMMON_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


SOURCE_PART: part2
PART_ROOT: /kaggle/input/datasets/minhdat27/aic2026-batch1-part2
COMMON_ROOT: /kaggle/input/datasets/minhdat27/aic2026-batch1-common
OUTPUT_DIR: /kaggle/working/processed


In [3]:
# Cell 3 — Hàm tiện ích
def detect_group(path_or_id: str | Path) -> Optional[str]:
    text = str(path_or_id).replace("\\", "/").lower()
    for group in ASSIGNED_GROUPS:
        if group.lower() in text:
            return group
    return None

def keyframe_number(path: Path) -> Optional[int]:
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else None

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def first_existing(candidates):
    for p in candidates:
        if p is not None and p.exists():
            return p
    return None

def common_paths(video_id: str, keyframe_id: Optional[str] = None):
    media = first_existing([
        MEDIA_DIR / f"{video_id}.json",
        MEDIA_DIR / video_id / f"{video_id}.json",
    ])
    clip = first_existing([
        CLIP_DIR / f"{video_id}.npy",
        CLIP_DIR / video_id / f"{video_id}.npy",
    ])
    mapping = first_existing([
        MAP_DIR / f"{video_id}.csv",
        MAP_DIR / video_id / f"{video_id}.csv",
    ])
    obj = None
    if keyframe_id is not None:
        obj = first_existing([
            OBJECTS_DIR / video_id / f"{keyframe_id}.json",
            OBJECTS_DIR / f"{video_id}_{keyframe_id}.json",
        ])
    return media, clip, mapping, obj

def safe_json(path: Optional[Path]) -> dict:
    if path is None:
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}

def pick_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    columns = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in columns:
            return columns[candidate]
    return None

def safe_relative(path: Optional[Path], root: Path) -> Optional[str]:
    if path is None:
        return None
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)

FRAME_COLUMNS = [
    "frame_id", "frame_idx", "frame_index", "frame",
    "original_frame", "frame_number", "idx_frame"
]
TIME_COLUMNS = [
    "timestamp", "timestamp_sec", "time",
    "time_sec", "pts_time", "seconds"
]


In [4]:
# Cell 4 — Quét video và keyframe thuộc Person 3
video_files = natsorted([
    p for p in PART_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS and detect_group(p) in ASSIGNED_GROUPS
], key=str)

keyframe_files = natsorted([
    p for p in PART_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS and detect_group(p) in ASSIGNED_GROUPS
], key=str)

if MAX_VIDEOS is not None:
    video_files = video_files[:MAX_VIDEOS]
if MAX_KEYFRAMES is not None:
    keyframe_files = keyframe_files[:MAX_KEYFRAMES]

print(f"Videos: {len(video_files):,}")
print(f"Keyframes: {len(keyframe_files):,}")
print("Groups tìm thấy:", sorted({detect_group(p) for p in video_files + keyframe_files if detect_group(p)}))

Videos: 191
Keyframes: 36,378
Groups tìm thấy: ['L23', 'L24', 'L26_d', 'L29']


In [5]:
# Cell 5 — Kiểm tra dữ liệu và tạo manifest retrieval-ready
videos_data, manifest_data, feature_catalog_data, errors_data = [], [], [], []

# Gom keyframe theo video.
keyframes_by_video = {}
for image_path in keyframe_files:
    keyframes_by_video.setdefault(image_path.parent.name, []).append(image_path)

for video_path in tqdm(video_files, desc="Processing videos"):
    video_id = video_path.stem
    group = detect_group(video_path) or detect_group(video_id)

    cap = cv2.VideoCapture(str(video_path))
    open_ok = cap.isOpened()
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if open_ok else 0.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if open_ok else 0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) if open_ok else 0
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) if open_ok else 0
    cap.release()

    if not open_ok:
        errors_data.append({
            "severity": "error",
            "stage": "video",
            "video_id": video_id,
            "asset": str(video_path),
            "error_type": "VIDEO_READ_ERROR",
            "detail": "Không thể mở video",
        })

    media_path, clip_path, map_path, _ = common_paths(video_id)
    metadata = safe_json(media_path)

    # Đọc mapping.
    map_df = pd.DataFrame()
    if map_path is not None:
        try:
            map_df = pd.read_csv(map_path)
        except Exception as exc:
            errors_data.append({
                "severity": "error",
                "stage": "mapping",
                "video_id": video_id,
                "asset": str(map_path),
                "error_type": "MAP_READ_ERROR",
                "detail": str(exc),
            })

    frame_col = pick_column(map_df, FRAME_COLUMNS) if not map_df.empty else None
    time_col = pick_column(map_df, TIME_COLUMNS) if not map_df.empty else None

    # Đọc shape CLIP bằng mmap, không nạp toàn bộ vector vào RAM.
    feature_count = None
    embedding_dim = None
    feature_dtype = None
    if clip_path is not None:
        try:
            arr = np.load(clip_path, mmap_mode="r", allow_pickle=False)
            if arr.ndim != 2:
                raise ValueError(f"Expected 2D CLIP array, got {arr.shape}")
            feature_count = int(arr.shape[0])
            embedding_dim = int(arr.shape[1])
            feature_dtype = str(arr.dtype)
        except Exception as exc:
            errors_data.append({
                "severity": "error",
                "stage": "feature",
                "video_id": video_id,
                "asset": str(clip_path),
                "error_type": "FEATURE_READ_ERROR",
                "detail": str(exc),
            })

    keyframes = natsorted(keyframes_by_video.get(video_id, []), key=str)
    keyframe_count = len(keyframes)
    map_count = len(map_df)

    counts = [keyframe_count, map_count]
    if feature_count is not None:
        counts.append(feature_count)
    aligned_count = min(counts) if counts else 0
    alignment_ok = len(set(counts)) == 1

    if not alignment_ok:
        errors_data.append({
            "severity": "error",
            "stage": "alignment",
            "video_id": video_id,
            "asset": None,
            "error_type": "COUNT_MISMATCH",
            "detail": (
                f"keyframes={keyframe_count}, map_rows={map_count}, "
                f"clip_vectors={feature_count}"
            ),
        })

    for required_path, error_type in [
        (clip_path, "MISSING_CLIP_FEATURE"),
        (map_path, "MISSING_MAP_CSV"),
    ]:
        if required_path is None:
            errors_data.append({
                "severity": "error",
                "stage": "common_link",
                "video_id": video_id,
                "asset": None,
                "error_type": error_type,
                "detail": "Thiếu tài nguyên Common bắt buộc",
            })

    if media_path is None:
        errors_data.append({
            "severity": "warning",
            "stage": "common_link",
            "video_id": video_id,
            "asset": None,
            "error_type": "MISSING_MEDIA_INFO",
            "detail": "Một số video có thể không có media-info",
        })

    videos_data.append({
        "video_id": video_id,
        "source_part": SOURCE_PART,
        "group_id": group,
        "video_path": str(video_path),
        "video_relpath": safe_relative(video_path, PART_ROOT),
        "open_ok": open_ok,
        "fps": round(fps, 6),
        "total_frames": total_frames,
        "width": width,
        "height": height,
        "duration_sec": round(total_frames / fps, 6) if fps > 0 else None,
        "keyframe_count": keyframe_count,
        "map_count": map_count,
        "feature_count": feature_count,
        "embedding_dim": embedding_dim,
        "feature_dtype": feature_dtype,
        "alignment_ok": alignment_ok,
        "clip_feature_path": str(clip_path) if clip_path else None,
        "clip_feature_relpath": safe_relative(clip_path, COMMON_ROOT),
        "map_csv_path": str(map_path) if map_path else None,
        "map_csv_relpath": safe_relative(map_path, COMMON_ROOT),
        "media_info_path": str(media_path) if media_path else None,
        "media_info_relpath": safe_relative(media_path, COMMON_ROOT),
        "title": metadata.get("title"),
        "author": metadata.get("author"),
        "publish_date": metadata.get("publish_date"),
        "watch_url": metadata.get("watch_url"),
    })

    feature_catalog_data.append({
        "video_id": video_id,
        "source_part": SOURCE_PART,
        "clip_feature_path": str(clip_path) if clip_path else None,
        "clip_feature_relpath": safe_relative(clip_path, COMMON_ROOT),
        "feature_count": feature_count,
        "embedding_dim": embedding_dim,
        "feature_dtype": feature_dtype,
        "alignment_ok": alignment_ok,
    })

    for feature_row in range(aligned_count):
        image_path = keyframes[feature_row]
        keyframe_id = image_path.stem
        map_row = map_df.iloc[feature_row]

        raw_frame = map_row.get(frame_col) if frame_col is not None else None
        raw_time = map_row.get(time_col) if time_col is not None else None

        try:
            frame_id = int(float(raw_frame)) if pd.notna(raw_frame) else None
        except Exception:
            frame_id = None

        # Chỉ fallback về số trong tên keyframe khi CSV không có frame column.
        if frame_id is None:
            frame_id = keyframe_number(image_path)

        try:
            timestamp_sec = float(raw_time) if pd.notna(raw_time) else None
        except Exception:
            timestamp_sec = None
        if timestamp_sec is None and frame_id is not None and fps > 0:
            timestamp_sec = frame_id / fps

        _, _, _, object_path = common_paths(video_id, keyframe_id)
        image_ok = cv2.imread(str(image_path)) is not None

        if not image_ok:
            errors_data.append({
                "severity": "error",
                "stage": "keyframe",
                "video_id": video_id,
                "asset": str(image_path),
                "error_type": "IMAGE_READ_ERROR",
                "detail": "Không thể đọc keyframe",
            })

        manifest_data.append({
            "global_key": f"{SOURCE_PART}:{video_id}:{feature_row}",
            "source_part": SOURCE_PART,
            "group_id": group,
            "video_id": video_id,
            "feature_row": feature_row,
            "keyframe_order": feature_row,
            "keyframe_id": keyframe_id,
            "keyframe_number": keyframe_number(image_path),
            "frame_id": frame_id,
            "timestamp_sec": timestamp_sec,
            "embedding_dim": embedding_dim,
            "feature_dtype": feature_dtype,
            "video_path": str(video_path),
            "video_relpath": safe_relative(video_path, PART_ROOT),
            "keyframe_path": str(image_path),
            "keyframe_relpath": safe_relative(image_path, PART_ROOT),
            "clip_feature_path": str(clip_path) if clip_path else None,
            "clip_feature_relpath": safe_relative(clip_path, COMMON_ROOT),
            "map_csv_path": str(map_path) if map_path else None,
            "map_csv_relpath": safe_relative(map_path, COMMON_ROOT),
            "object_path": str(object_path) if object_path else None,
            "object_relpath": safe_relative(object_path, COMMON_ROOT),
            "media_info_path": str(media_path) if media_path else None,
            "media_info_relpath": safe_relative(media_path, COMMON_ROOT),
            "title": metadata.get("title"),
            "author": metadata.get("author"),
            "publish_date": metadata.get("publish_date"),
            "watch_url": metadata.get("watch_url"),
            "image_ok": image_ok,
            "has_object": object_path is not None,
            "has_media_info": media_path is not None,
            "alignment_ok": alignment_ok,
        })


Processing videos:   0%|          | 0/191 [00:00<?, ?it/s]

In [6]:
# Cell 6 — Tạo bảng và xuất cùng một form
df_videos = pd.DataFrame(videos_data).sort_values("video_id").reset_index(drop=True)
df_manifest = (
    pd.DataFrame(manifest_data)
    .sort_values(["video_id", "feature_row"])
    .reset_index(drop=True)
)
df_features = (
    pd.DataFrame(feature_catalog_data)
    .sort_values("video_id")
    .reset_index(drop=True)
)
df_errors = pd.DataFrame(
    errors_data,
    columns=[
        "severity", "stage", "video_id", "asset",
        "error_type", "detail",
    ],
)

# global_id chỉ có ý nghĩa trong output của Part hiện tại.
# Sau khi merge 4 Part, notebook merge sẽ đánh lại global_id toàn cục.
df_manifest.insert(0, "global_id", np.arange(len(df_manifest), dtype=np.int64))

manifest_path = OUTPUT_DIR / "manifest_keyframes.parquet"
videos_path = OUTPUT_DIR / "manifest_videos.parquet"
features_path = OUTPUT_DIR / "feature_catalog.parquet"
errors_path = OUTPUT_DIR / "errors.csv"

df_manifest.to_parquet(
    manifest_path, index=False, compression="zstd"
)
df_videos.to_parquet(
    videos_path, index=False, compression="zstd"
)
df_features.to_parquet(
    features_path, index=False, compression="zstd"
)
df_errors.to_csv(
    errors_path, index=False, encoding="utf-8-sig"
)

checksum_rows = []
for path in [manifest_path, videos_path, features_path, errors_path]:
    checksum_rows.append({
        "file": path.name,
        "sha256": sha256_file(path) if CALCULATE_SHA256 else None,
        "size_bytes": path.stat().st_size,
    })

df_checksums = pd.DataFrame(checksum_rows)
df_checksums.to_csv(
    OUTPUT_DIR / "checksums.csv",
    index=False,
    encoding="utf-8-sig",
)

display(df_manifest.head())


,global_id,global_key,source_part,group_id,video_id,feature_row,keyframe_order,keyframe_id,keyframe_number,frame_id,...,media_info_path,media_info_relpath,title,author,publish_date,watch_url,image_ok,has_object,has_media_info,alignment_ok
0,0,part2:L23_V001:0,part2,L23,L23_V001,0,0,001,1,0,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L23_V001.json,DIỄN BIẾN VÒNG ĐUA CUỐI CÙNG - CHẶNG 1 CÚP TRU...,HTV Sports,02/04/2024,https://youtube.com/watch?v=3oVaFPFt9vI,True,True,True,True
1,1,part2:L23_V001:1,part2,L23,L23_V001,1,1,002,2,87,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L23_V001.json,DIỄN BIẾN VÒNG ĐUA CUỐI CÙNG - CHẶNG 1 CÚP TRU...,HTV Sports,02/04/2024,https://youtube.com/watch?v=3oVaFPFt9vI,True,True,True,True
2,2,part2:L23_V001:2,part2,L23,L23_V001,2,2,003,3,125,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L23_V001.json,DIỄN BIẾN VÒNG ĐUA CUỐI CÙNG - CHẶNG 1 CÚP TRU...,HTV Sports,02/04/2024,https://youtube.com/watch?v=3oVaFPFt9vI,True,True,True,True
3,3,part2:L23_V001:3,part2,L23,L23_V001,3,3,004,4,222,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L23_V001.json,DIỄN BIẾN VÒNG ĐUA CUỐI CÙNG - CHẶNG 1 CÚP TRU...,HTV Sports,02/04/2024,https://youtube.com/watch?v=3oVaFPFt9vI,True,True,True,True
4,4,part2:L23_V001:4,part2,L23,L23_V001,4,4,005,5,250,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L23_V001.json,DIỄN BIẾN VÒNG ĐUA CUỐI CÙNG - CHẶNG 1 CÚP TRU...,HTV Sports,02/04/2024,https://youtube.com/watch?v=3oVaFPFt9vI,True,True,True,True


In [7]:
# Cell 7 — Summary, schema và đóng gói
required_columns = [
    "global_id", "global_key", "source_part", "video_id",
    "feature_row", "frame_id", "keyframe_path", "video_path",
    "clip_feature_path", "embedding_dim",
]

acceptance = {
    "manifest_has_required_columns": all(
        c in df_manifest.columns for c in required_columns
    ),
    "global_key_is_unique": bool(df_manifest["global_key"].is_unique),
    "video_feature_row_is_unique": bool(
        ~df_manifest.duplicated(["video_id", "feature_row"]).any()
    ),
    "all_videos_readable": bool(df_videos["open_ok"].all())
        if not df_videos.empty else False,
    "all_keyframes_readable": bool(df_manifest["image_ok"].all())
        if not df_manifest.empty else False,
    "all_video_alignment_ok": bool(df_videos["alignment_ok"].all())
        if not df_videos.empty else False,
    "all_embeddings_are_512d": bool(
        df_manifest["embedding_dim"].dropna().eq(512).all()
    ),
}

summary = {
    "schema_version": "aic2026-single-part-v1",
    "source_part": SOURCE_PART,
    "assigned_groups": ASSIGNED_GROUPS,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_videos": int(len(df_videos)),
    "total_keyframes": int(len(df_manifest)),
    "total_feature_files": int(len(df_features)),
    "total_errors": int(
        (df_errors["severity"] == "error").sum()
        if not df_errors.empty else 0
    ),
    "total_warnings": int(
        (df_errors["severity"] == "warning").sum()
        if not df_errors.empty else 0
    ),
    "acceptance": acceptance,
}

schema = {
    "schema_version": "aic2026-single-part-v1",
    "main_file": "manifest_keyframes.parquet",
    "primary_key_before_merge": "global_key",
    "vector_locator": [
        "clip_feature_relpath",
        "feature_row",
    ],
    "submission_mapping": [
        "video_id",
        "frame_id",
    ],
    "preview_mapping": [
        "keyframe_relpath",
        "video_relpath",
        "timestamp_sec",
    ],
    "manifest_keyframes_columns": {
        col: str(dtype)
        for col, dtype in df_manifest.dtypes.items()
    },
    "manifest_videos_columns": {
        col: str(dtype)
        for col, dtype in df_videos.dtypes.items()
    },
    "feature_catalog_columns": {
        col: str(dtype)
        for col, dtype in df_features.dtypes.items()
    },
}

with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with (OUTPUT_DIR / "schema.json").open("w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(
    f"/kaggle/working/aic2026_{SOURCE_PART}_processed",
    "zip",
    root_dir=OUTPUT_DIR,
)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nCác file output chuẩn:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)
print("ZIP:", zip_path)


{
  "schema_version": "aic2026-single-part-v1",
  "source_part": "part2",
  "assigned_groups": [
    "L23",
    "L24",
    "L26_d",
    "L29"
  ],
  "created_at_utc": "2026-08-02T17:09:59.237032+00:00",
  "total_videos": 191,
  "total_keyframes": 36378,
  "total_feature_files": 191,
  "total_errors": 0,
  "total_warnings": 0,
  "acceptance": {
    "manifest_has_required_columns": true,
    "global_key_is_unique": true,
    "video_feature_row_is_unique": true,
    "all_videos_readable": true,
    "all_keyframes_readable": true,
    "all_video_alignment_ok": true,
    "all_embeddings_are_512d": true
  }
}

Các file output chuẩn:
 - checksums.csv
 - errors.csv
 - feature_catalog.parquet
 - manifest_keyframes.parquet
 - manifest_videos.parquet
 - schema.json
 - summary.json
ZIP: /kaggle/working/aic2026_part2_processed.zip


## Output chuẩn cho mọi Part

Mỗi thành viên phải bàn giao đúng bộ file sau:

```text
manifest_keyframes.parquet
manifest_videos.parquet
feature_catalog.parquet
errors.csv
checksums.csv
summary.json
schema.json
```

Không đổi tên file theo `p1`, `p2`, `p3`, `p4`. Dataset output trên Kaggle đã giúp phân biệt Part.

### File dùng cho retrieval

- `manifest_keyframes.parquet`: một dòng/keyframe/vector.
- `clip_feature_relpath + feature_row`: xác định vector trong `.npy`.
- `video_id + frame_id`: tạo câu trả lời nộp BTC.
- `keyframe_relpath + video_relpath + timestamp_sec`: hiển thị và preview.
- `source_part`: xác định dataset vật lý chứa video/keyframe.

Sau khi có đủ 4 output, chỉ cần concat bốn file
`manifest_keyframes.parquet`, kiểm tra trùng `global_key`, rồi đánh lại `global_id`.
